# Running on hardware: IQM Resonance

This notebook is the **IQM Resonance** connection guide — the IQM-ecosystem replacement
for the IBM `QiskitRuntimeService` workflow. It shows how to install the client, obtain an
API token, connect to a **Garnet** (20-qubit) or **Emerald** (54-qubit) quantum computer,
and run a circuit — plus a fully offline local alternative that needs no token.

> You do **not** need to run this notebook to use the rest of the IQM folder: the helper
> `iqm_backend.py` already wraps every step below behind `get_backend` / `run_counts`.
> This guide simply unpacks what that helper does under the hood.


## 1. Install the IQM Qiskit client

```bash
pip install "qiskit-iqm>=18"
```

`qiskit-iqm` provides `IQMProvider` (cloud access to IQM Resonance), `transpile_to_IQM`
(transpilation to the IQM native gate set), and local fake backends such as `IQMFakeApollo`.


## 2. Get an API token and set `IQM_TOKEN`

1. Log in to the IQM Resonance dashboard: **https://resonance.iqm.tech/**
2. Create / copy a personal **API token**.
3. Expose it to this session through the `IQM_TOKEN` environment variable.

**bash / zsh (Linux, macOS):**

```bash
export IQM_TOKEN="<your-resonance-token>"
```

**PowerShell (Windows):**

```powershell
$env:IQM_TOKEN = "<your-resonance-token>"
```

Set the variable **before** launching Jupyter so the kernel inherits it. Alternatively you
can pass the token explicitly via `get_backend(target, token=...)` from `iqm_backend.py`, but the env-var approach keeps
secrets out of the notebook.


## 3. Connect to an IQM quantum computer

Pick the machine with the `quantum_computer` argument: `"garnet"` (20q real QPU),
`"emerald"` (54q real QPU), or a cloud mock such as `"garnet:mock"` (checks the job pipeline, returns random bitstrings).
`IQMProvider` reads the token from the `IQM_TOKEN` environment variable automatically.


In [ ]:
from iqm.qiskit_iqm import IQMProvider, transpile_to_IQM

# IQMProvider(url, quantum_computer=...) talks to IQM Resonance; the token is read from IQM_TOKEN.
provider = IQMProvider("https://resonance.iqm.tech", quantum_computer="garnet")  # or "emerald", "garnet:mock"
backend = provider.get_backend()                   # a Qiskit backend object: has .run(), .num_qubits, .coupling_map, ...
print(backend.name, backend.num_qubits)


## 4. Build, transpile, and run a Bell-pair circuit

Build a standard two-qubit Bell circuit, transpile it to the IQM native gate set with
`transpile_to_IQM`, submit it with `backend.run`, and read the measurement counts.


In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)         # QuantumCircuit(number_of_qubits, number_of_classical_bits)
qc.h(0)                           # Hadamard on qubit 0 -> superposition
qc.cx(0, 1)                       # CNOT, control qubit 0, target qubit 1 -> entangled Bell pair
qc.measure([0, 1], [0, 1])        # measure(qubits, classical_bits)
display(qc.draw('mpl'))           # display(): show a figure that is not the last line of the cell

# Rewrite the circuit with the IQM native gates (PRX rotations 'r' + CZ) and map it onto the chip's qubits
tqc = transpile_to_IQM(qc, backend=backend, optimization_level=3)
display(tqc.draw('mpl', idle_wires=False))   # idle_wires=False hides the chip qubits the circuit does not use

job = backend.run(tqc, shots=1000)           # submit; shots = number of repetitions
print(job.result().get_counts())             # dict {'bitstring': how many shots gave it}


## 5. Available targets

| Target | Runs on | Qubits | Token? | QPU credits? |
|---|---|---|---|---|
| `fake_apollo`  | local noisy simulator (offline) | 20 | no  | no  |
| `garnet:mock`  | cloud mock of Garnet (pipeline check: returns RANDOM bitstrings, no physics)  | 20 | yes | no  |
| `emerald:mock` | cloud mock of Emerald (pipeline check: returns RANDOM bitstrings, no physics) | 54 | yes | no  |
| `garnet`       | **real Garnet QPU**  | 20 | yes | yes |
| `emerald`      | **real Emerald QPU** | 54 | yes | yes |

Develop and validate logic with `fake_apollo` (free, offline), check the cloud pipeline on a `:mock` target,
then switch to `garnet` / `emerald` for a real-hardware run.


## 6. Local offline alternative (no token)

For development and CI you can skip the cloud entirely and use a local fake backend that
mimics Garnet's topology and noise. No token, no network, no QPU credits.


In [ ]:
from iqm.qiskit_iqm import IQMFakeApollo

backend = IQMFakeApollo()         # local noisy simulator with Garnet-like topology and noise model
print(backend.name, backend.num_qubits)

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

tqc = transpile_to_IQM(qc, backend=backend, optimization_level=3)
display(tqc.draw('mpl', idle_wires=False))   # the same Bell pair in native PRX + CZ gates
print(backend.run(tqc, shots=1000).result().get_counts())


## 7. IQM native gates and the Crystal topology

IQM Garnet and Emerald are **Crystal**-architecture superconducting chips laid out on a
**square-lattice** topology (nearest-neighbour coupling on a 2-D grid). Their native gate
set is:

- **PRX** — the phased-RX single-qubit rotation `R(theta, phi)`.
- **CZ** — the two-qubit controlled-Z entangling gate.

`transpile_to_IQM` rewrites any Qiskit circuit (`h`, `cx`, `rz`, ...) into this `{PRX, CZ}`
basis and routes it onto the square-lattice coupling map.

The helper module **`iqm_backend.py`** in this folder wraps everything above:

```python
from iqm_backend import get_backend, run_counts

# get_backend picks the right local/cloud backend from a single target string
backend = get_backend("fake_apollo")      # or "garnet:mock" / "garnet" / "emerald"

# run_counts = transpile_to_IQM + backend.run + get_counts, in one call
counts = run_counts(qc, "fake_apollo", shots=2048)
```

Use the helper for the rest of the course notebooks; use this guide when you need to see or
customise the raw `IQMProvider` / `transpile_to_IQM` / `backend.run` steps directly.
